# 🎓 Pipeline Deteksi Orisinalitas - Tuning Spesifik Mata Kuliah

Notebook ini dikonfigurasi secara khusus untuk memproses data dan melakukan pelatihan ulang (*fine-tuning*) model SBERT yang difokuskan pada **satu pembahasan mata kuliah tertentu**. Pendekatan ini berguna untuk melakukan penyesuaian (*tuning*) semantik lokal secara bertahap per mata kuliah.

---
## 🛠️ Langkah 1: Setup GPU & Instalasi Pustaka (*Dependency Setup*)

Sebelum menjalankan sel di bawah, pastikan runtime Colab menggunakan akselerasi GPU (T4 GPU):
1. Pilih menu **Runtime** > **Change runtime type**.
2. Setel **Hardware accelerator** ke **GPU** lalu klik **Save**.

In [ ]:
# Pemasangan pustaka pendukung pipeline (Membatasi qdrant-client >= 1.9.0, < 1.16.0 untuk mencegah hang Pydantic V2 di Python 3.12)
!pip install -q pymupdf easyocr sentence-transformers "qdrant-client>=1.9.0,<1.16.0" pydantic pydantic-settings pillow torch transformers psycopg2-binary

---
## 🔑 Langkah 2: Konfigurasi Variabel Lingkungan & Kredensial secara Aman

Masukkan kredensial berikut pada panel **Secrets** (ikon kunci 🔑) di bagian kiri Google Colab, lalu aktifkan akses notebook:
* $QDRANT_URL$
* $QDRANT_API_KEY$
* $DATABASE_URL$
* $HF_TOKEN$ *(Opsional - untuk akses Hugging Face)*

Jalankan cell di bawah untuk memuat konfigurasi tersebut.

In [ ]:
import os
from google.colab import userdata

# Membaca kredensial dari Google Colab Secrets secara dinamis dan membersihkan spasi/newline (\r\n)
os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL").strip()
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY").strip()
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL").strip()

# Membaca HF_TOKEN dari Google Colab Secrets jika tersedia
hf_token = userdata.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token.strip()
    print("HF_TOKEN berhasil dimuat dari Colab Secrets.")
else:
    print("HF_TOKEN tidak ditemukan di Colab Secrets. Lanjutkan tanpa token Hugging Face.")

print("Konfigurasi lingkungan berhasil dimuat secara aman.")

---
## 📂 Langkah 3: Pemasangan Google Drive & Fokus Folder Mata Kuliah

Di bawah ini disetel konfigurasi khusus untuk membatasi ruang lingkup kerja pada satu mata kuliah terpilih. Pengguna dapat mengubah nama folder pada variabel $MATA_KULIAH_FOLDER$ dan memasukkan ID referensi pada $MATA_KULIAH_ID$.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# =========================================================================
# KONFIGURASI FOKUS MATA KULIAH
# =========================================================================
# Ganti nama folder sesuai mata kuliah yang ingin dieksekusi/dituning
MATA_KULIAH_FOLDER = "Laporan DPP 2025" 

# ID UUID Mata Kuliah yang terdaftar di database PostgreSQL (tabel mata_kuliah)
# Ganti dengan ID mata kuliah target
MATA_KULIAH_ID = "77b21e8d-d779-46f3-a128-091a13be45cb"
# =========================================================================

# Direktori kerja spesifik mata kuliah
WORKSPACE_DIR = f"/content/drive/MyDrive/DATASET (cloud laporan matakuliah)/{MATA_KULIAH_FOLDER}"
os.makedirs(WORKSPACE_DIR, exist_ok=True)
print(f"Direktori kerja dibatasi pada folder mata kuliah: {WORKSPACE_DIR}")
print(f"ID Mata Kuliah terpilih untuk tuning: {MATA_KULIAH_ID}")

---
## 📄 Langkah 4: Modul Ekstraksi Dokumen & Fallback OCR

Modul untuk membongkar berkas PDF. Jika halaman PDF berupa dokumen hasil pemindaian (*scanned PDF*), sistem memicu EasyOCR dengan pemanfaatan akselerasi GPU secara otomatis.

In [ ]:
import fitz  # PyMuPDF
import easyocr
import re
from PIL import Image
import io

class DocumentParser:
    def __init__(self):
        self.reader = None

    def _get_ocr_reader(self):
        if self.reader is None:
            self.reader = easyocr.Reader(['id', 'en'], gpu=True)
        return self.reader

    def parse_pdf(self, pdf_path: str, document_id: str):
        doc = fitz.open(pdf_path)
        extracted_text = ""
        words_with_geometry = []
        extracted_images = []
        
        total_chars = sum(len(page.get_text()) for page in doc)
        is_scanned = total_chars <= 50

        print(f"Status Dokumen: {'SCANNED (Memicu Fallback OCR)' if is_scanned else 'DIGITAL (Ekstraksi Teks Langsung)'}")

        for page_idx, page in enumerate(doc):
            page_text = ""
            
            if is_scanned:
                zoom = 150 / 72.0
                mat = fitz.Matrix(zoom, zoom)
                pix = page.get_pixmap(matrix=mat)
                img_data = pix.tobytes("png")
                
                reader = self._get_ocr_reader()
                results = reader.readtext(img_data)
                
                scale = 72.0 / 150.0
                for bbox, text, confidence in results:
                    x1 = min(pt[0] for pt in bbox) * scale
                    y1 = min(pt[1] for pt in bbox) * scale
                    x2 = max(pt[0] for pt in bbox) * scale
                    y2 = max(pt[1] for pt in bbox) * scale
                    
                    words_with_geometry.append({
                        "text": text,
                        "geometry": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
                        "page": page_idx + 1
                    })
                    page_text += text + " "
            else:
                words = page.get_text("words")
                for w in words:
                    words_with_geometry.append({
                        "text": w[4],
                        "geometry": {"x1": w[0], "y1": w[1], "x2": w[2], "y2": w[3]},
                        "page": page_idx + 1
                    })
                    page_text += w[4] + " "
            
            extracted_text += page_text + "\n"

            # Ekstraksi Gambar
            for img_info in page.get_images(full=True):
                xref = img_info[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                
                image = Image.open(io.BytesIO(image_bytes))
                if image.width >= 150 and image.height >= 150:
                    image_name = f"img_{document_id}_p{page_idx+1}_{xref}.png"
                    extracted_images.append({
                        "filename": image_name,
                        "bytes": image_bytes,
                        "page": page_idx + 1
                    })
                    
        return {
            "text": extracted_text,
            "words": words_with_geometry,
            "images": extracted_images
        }

parser = DocumentParser()
print("Modul Parser Dokumen siap.")

---
## 🧬 Langkah 5: Pemotongan Teks & Generasi Dense Text Semantics Embedding

Menghasilkan vektor semantik 768 dimensi menggunakan model **`intfloat/multilingual-e5-base`** dengan pembagian potongan teks berukuran 250 kata.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

class TextEmbedder:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer("intfloat/multilingual-e5-base", device=self.device)
        print(f"Model SBERT dimuat pada perangkat: {self.device.upper()}")

    def chunk_text(self, words_with_geometry: list, chunk_size: int = 250, overlap: int = 50) -> list:
        chunks = []
        step = chunk_size - overlap
        
        for i in range(0, len(words_with_geometry), step):
            chunk_words = words_with_geometry[i:i + chunk_size]
            if not chunk_words:
                break
                
            chunk_text = " ".join([w["text"] for w in chunk_words])
            
            x1 = min(w["geometry"]["x1"] for w in chunk_words)
            y1 = min(w["geometry"]["y1"] for w in chunk_words)
            x2 = max(w["geometry"]["x2"] for w in chunk_words)
            y2 = max(w["geometry"]["y2"] for w in chunk_words)
            page = chunk_words[0]["page"]
            
            chunks.append({
                "text": chunk_text,
                "geometry": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
                "page": page
            })
        return chunks

    def generate_embeddings(self, chunks: list) -> list:
        texts_to_embed = [f"passage: {c['text']}" for c in chunks]
        embeddings = self.model.encode(texts_to_embed, batch_size=32, convert_to_numpy=True)
        
        for idx, emb in enumerate(embeddings):
            chunks[idx]["embedding"] = emb.tolist()
        return chunks

text_embedder = TextEmbedder()

---
## 🖼️ Langkah 6: Generasi Multimodal Image Embedding (Self-Contained Imports)

Mengubah citra laporan menjadi visual embedding 512 dimensi menggunakan model CLIP. Pustaka `torch`, `PIL.Image`, dan `io` diimpor langsung secara eksplisit di dalam sel ini untuk mencegah NameError apabila sel dijalankan secara terpisah.

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import io

class ImageEmbedder:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(self.device)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        print(f"Model CLIP dimuat pada perangkat: {self.device.upper()}")

    def generate_image_embedding(self, img_bytes: bytes) -> list:
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            
            # Penanganan adaptif jika model mengembalikan objek BaseModelOutputWithPooling (transformers >= 5.x)
            if not isinstance(image_features, torch.Tensor):
                if hasattr(image_features, "image_embeds") and image_features.image_embeds is not None:
                    image_features = image_features.image_embeds
                elif hasattr(image_features, "pooler_output") and image_features.pooler_output is not None:
                    image_features = image_features.pooler_output
                elif hasattr(image_features, "last_hidden_state") and image_features.last_hidden_state is not None:
                    image_features = image_features.last_hidden_state
            
            # Normalisasi L2 untuk menghitung Cosine Similarity via Dot Product
            normalized_features = torch.nn.functional.normalize(image_features, p=2, dim=-1)
            
        return normalized_features.cpu().numpy()[0].tolist()

image_embedder = ImageEmbedder()

---
## 🗄️ Langkah 7: Pengunggahan Vektor ke Qdrant DB (Konfigurasi Named Vector)

Mengunggah vektor representasi ke Qdrant Cloud. Karena koleksi `laporan_text_vectors` menggunakan konfigurasi multi-vektor, komponen vektor teks bertipe dense harus dikirim menggunakan nama kunci `"text-dense"` untuk menghindari eror Bad Request.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qd_models
import os

class VectorRepository:
    def __init__(self):
        self.client = QdrantClient(
            url=os.environ["QDRANT_URL"],
            api_key=os.environ["QDRANT_API_KEY"]
        )
        print("Koneksi Qdrant Cloud berhasil.")
        
        # Inisialisasi indeks payload untuk pencarian filter yang efisien (menghindari error Index required)
        try:
            self.client.create_payload_index(
                collection_name="laporan_text_vectors",
                field_name="id_laporan",
                field_schema=qd_models.PayloadSchemaType.KEYWORD
            )
            self.client.create_payload_index(
                collection_name="laporan_text_vectors",
                field_name="id_mata_kuliah",
                field_schema=qd_models.PayloadSchemaType.KEYWORD
            )
            print("Payload indexes (id_laporan & id_mata_kuliah) berhasil diverifikasi/dibuat.")
        except Exception as e:
            print(f"⚠️ Gagal membuat payload indexes otomatis: {str(e)}")

    def upload_text_vectors(self, document_id: str, embedded_chunks: list):
        points = []
        for idx, chunk in enumerate(embedded_chunks):
            points.append(
                qd_models.PointStruct(
                    id=hash(f"{document_id}_t_{idx}") % (10**10),
                    # Menggunakan Named Vector 'text-dense' sesuai konfigurasi schema DB
                    vector={
                        "text-dense": chunk["embedding"]
                    },
                    payload={
                        "id_laporan": document_id,
                        "text": chunk["text"],
                        "page": chunk["page"],
                        "geometry": chunk["geometry"],
                        "id_mata_kuliah": MATA_KULIAH_ID
                    }
                )
            )
            
        self.client.upsert(
            collection_name="laporan_text_vectors",
            points=points
        )
        print(f"Mengunggah {len(points)} vektor teks ke Qdrant.")

    def upload_image_vectors(self, document_id: str, embedded_images: list):
        points = []
        for idx, img in enumerate(embedded_images):
            points.append(
                qd_models.PointStruct(
                    id=hash(f"{document_id}_i_{idx}") % (10**10),
                    # Koleksi gambar menggunakan Unnamed Vector (flat list)
                    vector=img["embedding"],
                    payload={
                        "id_laporan": document_id,
                        "filename": img["filename"],
                        "page": img["page"],
                        "id_mata_kuliah": MATA_KULIAH_ID
                    }
                )
            )
            
        self.client.upsert(
            collection_name="laporan_image_vectors",
            points=points
        )
        print(f"Mengunggah {len(points)} vektor citra ke Qdrant.")

vector_repo = VectorRepository()

---
## 🚀 Langkah 8: Pengujian Eksekusi Pipeline Spesifik Mata Kuliah (Batch Indexing dengan Proteksi Redundansi)

Menjalankan proses pengindeksan massal (*batch indexing*) untuk seluruh berkas PDF di dalam folder mata kuliah Google Drive. Sel ini dilengkapi dengan **proteksi redundansi otomatis** yang memeriksa apakah dokumen sudah pernah terindeks di Qdrant sebelum melakukan parsing dan ekstraksi.

In [ ]:
import os
import traceback
import re
from qdrant_client.http import models as qd_models

# 1. Mendapatkan daftar seluruh berkas PDF di dalam folder mata kuliah
if os.path.exists(WORKSPACE_DIR):
    pdf_files = [f for f in os.listdir(WORKSPACE_DIR) if f.endswith('.pdf')]
    print(f"Ditemukan {len(pdf_files)} berkas PDF di folder: {WORKSPACE_DIR}\n")
else:
    pdf_files = []
    print(f"Jalur direktori tidak ditemukan: {WORKSPACE_DIR}")

# 2. Proses iterasi pengindeksan massal
success_count = 0
failed_files = []

for idx, filename in enumerate(pdf_files):
    pdf_path = os.path.join(WORKSPACE_DIR, filename)
    
    # ID Laporan unik otomatis
    document_id = re.sub(r'[^a-zA-Z0-9_]', '_', filename[:-4])
    
    print(f"[{idx + 1}/{len(pdf_files)}] Memproses berkas: {filename} (ID: {document_id})")
    
    try: 
        # a. Proteksi Redundansi: Cek apakah dokumen sudah pernah terindeks di Qdrant
        existing_points, _ = vector_repo.client.scroll(
            collection_name="laporan_text_vectors",
            scroll_filter=qd_models.Filter(
                must=[
                    qd_models.FieldCondition(
                        key="id_laporan",
                        match=qd_models.MatchValue(value=document_id)
                    )
                ]
            ),
            limit=1,
            with_payload=False,
            with_vectors=False
        )
        
        if existing_points:
            print(f"   ⚠️ Berkas {filename} sudah terindeks sebelumnya. Dilewati untuk menghindari redundansi.\n")
            success_count += 1
            continue
            
        # b. Ekstraksi dokumen PDF & OCR Fallback
        parsed_data = parser.parse_pdf(pdf_path, document_id)
        
        # c. Pemotongan kalimat (chunking)
        chunks = text_embedder.chunk_text(parsed_data["words"])
        if not chunks:
            print(f"   ⚠️ Berkas {filename} kosong atau tidak ada teks terdeteksi. Dilewati.")
            continue
            
        # d. Generasi representasi vektor semantik teks
        embedded_chunks = text_embedder.generate_embeddings(chunks)
        
        # e. Generasi representasi vektor visual citra jika ada
        embedded_images = []
        for img in parsed_data["images"]:
            emb_vec = image_embedder.generate_image_embedding(img["bytes"])
            img["embedding"] = emb_vec
            embedded_images.append(img)
            
        # f. Unggah ke Qdrant Cloud
        vector_repo.upload_text_vectors(document_id, embedded_chunks)
        if embedded_images:
            vector_repo.upload_image_vectors(document_id, embedded_images)
            
        print(f"   ✅ Berhasil mengindeks {filename}. Teks chunks: {len(embedded_chunks)}, Gambar: {len(embedded_images)}\n")
        success_count += 1
        
    except Exception as e:
        print(f"   ❌ Gagal memproses berkas {filename}. Error: {str(e)}")
        failed_files.append((filename, str(e)))
        traceback.print_exc()
        print("\n")

# 3. Laporan akhir
print("=========================================================================")
print(f"PROSES BATCH INDEXING SELESAI.")
print(f"Total Berkas Diproses: {len(pdf_files)}")
print(f"Sukses Terindeks     : {success_count} berkas")
print(f"Gagal Terindeks      : {len(failed_files)} berkas")
if failed_files:
    print("\nDaftar berkas yang gagal:")
    for f_name, err in failed_files:
        print(f"- {f_name}: {err}")
print("=========================================================================")

---
## 🧬 Langkah 9: Pelatihan Ulang Model (*Model Fine-Tuning*) & Pencatatan Aktivitas Supabase

Pada langkah ini, proses penarikan data dari database PostgreSQL dibatasi secara ketat pada ID laporan dari mata kuliah terpilih ($MATA\_KULIAH\_ID$). 

Setelah pelatihan ulang selesai dan model disimpan di Google Drive, program akan otomatis mencatat riwayat pembaruan bobot model ini ke dalam tabel audit **`aktivitas_admin`** di database relasional Supabase.

In [ ]:
import psycopg2
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, losses
import datetime
import os
import uuid
from urllib.parse import urlparse, unquote

def get_database_connection(database_url: str):
    clean_url = database_url
    if clean_url.startswith("postgresql://"):
        clean_url = clean_url[len("postgresql://"):]
    elif clean_url.startswith("postgres://"):
        clean_url = clean_url[len("postgres://"):]
        
    if "@" in clean_url:
        creds, host_info = clean_url.rsplit("@", 1)
    else:
        raise ValueError("Format database URL salah (tidak mengandung '@')")
        
    if ":" in creds:
        username, password = creds.split(":", 1)
    else:
        username = creds
        password = ""
        
    if "/" in host_info:
        host_port, database = host_info.split("/", 1)
    else:
        host_port = host_info
        database = "postgres"
        
    if ":" in host_port:
        hostname, port = host_port.rsplit(":", 1)
        port = int(port)
    else:
        hostname = host_port
        port = 5432
        
    return psycopg2.connect(
        database=database,
        user=unquote(username),
        password=unquote(password),
        host=hostname,
        port=port
    )

def get_mata_kuliah_name(database_url: str, id_mata_kuliah: str) -> str:
    try:
        conn = get_database_connection(database_url)
        cur = conn.cursor()
        cur.execute("SELECT nama_matkul FROM mata_kuliah WHERE id_mata_kuliah = %s;", (id_mata_kuliah,))
        row = cur.fetchone()
        cur.close()
        conn.close()
        if row:
            return str(row[0])
        return "Unknown"
    except Exception as e:
        print(f"⚠️ Gagal menarik nama mata kuliah: {str(e)}")
        return "Unknown"

def get_verified_laporan_ids_by_matkul(database_url: str, id_mata_kuliah: str) -> list[str]:
    try:
        conn = get_database_connection(database_url)
        cur = conn.cursor()
        # Mencetak info detail koneksi (sensor password)
        params = conn.get_dsn_parameters()
        print(f"Menarik ID laporan terverifikasi dari PostgreSQL...")
        print(f"  [Koneksi] Host: {params.get('host')}")
        print(f"  [Koneksi] Port: {params.get('port')}")
        print(f"  [Koneksi] Database: {params.get('dbname')}")
        print(f"  [Koneksi] User: {params.get('user')}")
        
        query = """
            SELECT id_laporan 
            FROM laporan 
            WHERE (skor_orisinalitas > 75.0 OR skor_orisinalitas_koreksi > 75.0) 
              AND apakah_diverifikasi = True 
              AND id_mata_kuliah = %s;
        """
        cur.execute(query, (id_mata_kuliah,))
        rows = cur.fetchall()
        cur.close()
        conn.close()
        ids = [str(row[0]) for row in rows]
        print(f"Berhasil menarik {len(ids)} ID Laporan dari database.")
        return ids
    except Exception as e:
        print(f"Gagal menarik data dari database relasional: {str(e)}")
        return []

def log_fine_tuning_to_supabase(database_url: str, save_path: str, id_mata_kuliah: str, nama_mata_kuliah: str):
    try:
        conn = get_database_connection(database_url)
        cur = conn.cursor()
        
        # 1. Cari ID Admin pertama di database untuk dicatat sebagai aktor aktivitas
        cur.execute("""
            SELECT p.id_pengguna, p.nama 
            FROM pengguna p
            JOIN profil_pengguna pp ON p.id_pengguna = pp.id_pengguna
            WHERE pp.peran = 'ADMIN'
            LIMIT 1;
        """)
        row = cur.fetchone()
        if not row:
            cur.execute("SELECT id_pengguna, nama FROM pengguna LIMIT 1;")
            row = cur.fetchone()
            
        if not row:
            print("⚠️ Tidak ada pengguna terdaftar untuk dicatat sebagai aktor audit trail.")
            cur.close()
            conn.close()
            return
            
        id_aktor, nama_aktor = row
        id_aktivitas = str(uuid.uuid4())
        jenis_tindakan = "FINE_TUNE"
        entitas_target = "SYSTEM_CONFIG"
        deskripsi = f"Fine-tuning model SBERT untuk pembahasan mata kuliah {nama_mata_kuliah} (ID: {id_mata_kuliah}) selesai. Bobot model baru disimpan di Google Drive: {save_path}"
        
        # 2. Sisipkan baris log aktivitas admin
        cur.execute("""
            INSERT INTO aktivitas_admin (
                id_aktivitas_admin, 
                id_aktor, 
                jenis_tindakan, 
                entitas_target, 
                deskripsi, 
                alamat_ip, 
                agen_pengguna,
                tanggal_dibuat
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, NOW());
        """, (
            id_aktivitas,
            id_aktor,
            jenis_tindakan,
            entitas_target,
            deskripsi,
            "127.0.0.1",
            "Google Colab (Jupyter Notebook)"
        ))
        
        conn.commit()
        cur.close()
        conn.close()
        print(f"✅ Log aktivitas fine-tuning berhasil dicatat di Supabase (Aktor: {nama_aktor}).")
    except Exception as e:
        print(f"⚠️ Gagal mencatat log aktivitas ke Supabase: {str(e)}")

def get_training_texts_from_qdrant(qdrant_client, laporan_ids: list[str]) -> list[str]:
    from qdrant_client.http import models as qd_models
    texts = []
    if not laporan_ids:
        return texts
        
    print(f"Menarik chunk teks dari Qdrant untuk {len(laporan_ids)} laporan terverifikasi...")
    for id_laporan in laporan_ids:
        try:
            points, _ = qdrant_client.scroll(
                collection_name="laporan_text_vectors",
                scroll_filter=qd_models.Filter(
                    must=[
                        qd_models.FieldCondition(
                            key="id_laporan",
                            match=qd_models.MatchValue(value=id_laporan)
                        )
                    ]
                ),
                limit=1000,
                with_payload=True,
                with_vectors=False
            )
            for p in points:
                txt = p.payload.get("text", "")
                if txt.strip():
                    texts.append(txt)
        except Exception as e:
            print(f"Gagal me-scroll chunk untuk laporan {id_laporan}: {str(e)}")
            
    return texts

# -------------------------------------------------------------------------
# Fallback dinamis jika runtime di-restart dan memori global terhapus
# -------------------------------------------------------------------------
if 'MATA_KULIAH_ID' not in globals():
    MATA_KULIAH_ID = "77b21e8d-d779-46f3-a128-091a13be45cb"
    print(f"MATA_KULIAH_ID dipulihkan ke default: {MATA_KULIAH_ID}")
    
if 'MATA_KULIAH_FOLDER' not in globals():
    MATA_KULIAH_FOLDER = "Laporan DPP 2025"
    print(f"MATA_KULIAH_FOLDER dipulihkan ke default: {MATA_KULIAH_FOLDER}")

if 'WORKSPACE_DIR' not in globals():
    WORKSPACE_DIR = f"/content/drive/MyDrive/DATASET (cloud laporan matakuliah)/{MATA_KULIAH_FOLDER}"
    print(f"WORKSPACE_DIR dipulihkan ke default: {WORKSPACE_DIR}")

qdrant_client_instance = None
if 'vector_repo' in globals() and vector_repo is not None:
    qdrant_client_instance = vector_repo.client
else:
    try:
        from qdrant_client import QdrantClient
        q_url = os.environ.get("QDRANT_URL")
        q_key = os.environ.get("QDRANT_API_KEY")
        if not q_url or not q_key:
            from google.colab import userdata
            q_url = userdata.get("QDRANT_URL").strip()
            q_key = userdata.get("QDRANT_API_KEY").strip()
            os.environ["QDRANT_URL"] = q_url
            os.environ["QDRANT_API_KEY"] = q_key
        qdrant_client_instance = QdrantClient(url=q_url, api_key=q_key)
        print("Koneksi Qdrant Cloud diinisialisasi secara mandiri di Langkah 9.")
    except Exception as e:
        print("⚠️ Gagal menginisialisasi Qdrant Client secara mandiri. Pastikan sel Langkah 7 telah dijalankan.")

db_url = os.environ.get("DATABASE_URL")
if not db_url:
    try:
        from google.colab import userdata
        db_url = userdata.get("DATABASE_URL").strip()
        os.environ["DATABASE_URL"] = db_url
        print("DATABASE_URL berhasil dipulihkan secara dinamis dari Colab Secrets.")
    except Exception:
        pass

# Pemuatan otomatis model SBERT jika instance text_embedder hilang dari memori
sbert_model = None
if 'text_embedder' in globals() and text_embedder is not None:
    sbert_model = text_embedder.model
else:
    try:
        from sentence_transformers import SentenceTransformer
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        sbert_model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)
        print(f"Model SBERT diinisialisasi secara mandiri pada perangkat: {device.upper()}")
    except Exception as e:
        print(f"⚠️ Gagal memulihkan model SBERT secara mandiri. Pastikan sel Langkah 5 telah dijalankan.")

if not db_url:
    print("❌ ERROR: DATABASE_URL tidak ditemukan di os.environ maupun Colab Secrets.")
    print("   Pastikan Anda telah memasukkan 'DATABASE_URL' di Secrets (ikon kunci 🔑) dan mengaktifkan akses notebook.")
    laporan_ids = []
    texts_dataset = []
    matkul_name = "Unknown"
else:
    matkul_name = get_mata_kuliah_name(db_url, MATA_KULIAH_ID)
    laporan_ids = get_verified_laporan_ids_by_matkul(db_url, MATA_KULIAH_ID)
    if qdrant_client_instance is not None:
        texts_dataset = get_training_texts_from_qdrant(qdrant_client_instance, laporan_ids)
    else:
        texts_dataset = []

if not texts_dataset:
    print("Mengaktifkan data latihan alternatif untuk cold-start...")
    texts_dataset = [
        "Representasi tipe data tumpukan atau stack menggunakan metode LIFO.",
        "Operasi push digunakan untuk menambahkan elemen baru ke dalam tumpukan.",
        "Operasi pop digunakan untuk mengambil elemen teratas dari tumpukan.",
        "Implementasi tumpukan dapat dilakukan dengan struktur data array atau list berkait.",
        "Pengecekan tumpukan kosong dilakukan dengan fungsi is_empty pada objek stack."
    ]

print(f"Jumlah teks latihan terfilter mata kuliah ({matkul_name}): {len(texts_dataset)}")

if sbert_model is not None:
    train_examples = [InputExample(texts=[text, text]) for text in texts_dataset]
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=min(16, len(train_examples)))
    train_loss = losses.MultipleNegativesRankingLoss(model=sbert_model)

    print(f"Memulai fine-tuning model semantik SBERT khusus untuk mata kuliah {MATA_KULIAH_FOLDER}...")
    sbert_model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=1,
        show_progress_bar=True
    )

    TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = f"{WORKSPACE_DIR}/fine_tuned_sbert_matkul_{TIMESTAMP}"
    sbert_model.save(save_path)
    print(f"Model hasil tuning mata kuliah berhasil disimpan di Google Drive: {save_path}")
    
    # Mencatat aktivitas ke database Supabase secara otomatis
    if db_url:
        log_fine_tuning_to_supabase(db_url, save_path, MATA_KULIAH_ID, matkul_name)
else:
    print("❌ ERROR: Model SBERT tidak tersedia untuk proses fine-tuning. Silakan jalankan sel Langkah 5 terlebih dahulu.")

---
## 🔍 Langkah 10: Pengujian Orisinalitas Laporan Tunggal dengan Model Fine-Tuned

Sel ini digunakan untuk memverifikasi tingkat orisinalitas dari **satu berkas laporan baru** terhadap database laporan terindeks menggunakan model SBERT hasil *fine-tuning*.

In [ ]:
import os
import fitz  # PyMuPDF
from qdrant_client import QdrantClient
from qdrant_client.http import models as qd_models
from sentence_transformers import SentenceTransformer
import torch
import glob

# =========================================================================
# KONFIGURASI SIMULASI PENGECEKAN TUNGGAL
# =========================================================================
# 1. Masukkan berkas PDF laporan tunggal baru yang ingin dicuji kesamaannya
TEST_PDF_PATH = ""  # Contoh: "/content/drive/MyDrive/DATASET (cloud laporan matakuliah)/Laporan_Baru.pdf"

# 2. Jalur model fine-tuning (kosongkan "" untuk otomatis memuat model terbaru di WORKSPACE_DIR)
FINE_TUNED_MODEL_PATH = ""
# =========================================================================

if 'MATA_KULIAH_ID' not in globals():
    MATA_KULIAH_ID = "77b21e8d-d779-46f3-a128-091a13be45cb"
    
if 'WORKSPACE_DIR' not in globals():
    WORKSPACE_DIR = "/content/drive/MyDrive/DATASET (cloud laporan matakuliah)/Laporan DPP 2025"

if not TEST_PDF_PATH:
    print("❌ ERROR: Harap isi TEST_PDF_PATH dengan path berkas PDF laporan baru untuk memulai pengujian tunggal.")
elif not os.path.exists(TEST_PDF_PATH):
    print(f"❌ ERROR: Berkas PDF tidak ditemukan pada path: {TEST_PDF_PATH}")
else:
    # a. Pencarian jalur model fine-tuned terbaru secara otomatis
    model_path = FINE_TUNED_MODEL_PATH
    if not model_path:
        model_folders = glob.glob(f"{WORKSPACE_DIR}/fine_tuned_sbert_matkul_*")
        if model_folders:
            model_path = max(model_folders, key=os.path.getmtime)
            print(f"🔎 Menggunakan model hasil fine-tuning terbaru: {model_path}")
        else:
            model_path = "intfloat/multilingual-e5-base"
            print(f"⚠️ Model fine-tuned tidak ditemukan. Menggunakan model dasar: {model_path}")
            
    # b. Muat model semantik
    print("⏳ Memuat model Sentence-Transformer...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    eval_model = SentenceTransformer(model_path, device=device)
    print(f"✅ Model berhasil dimuat pada perangkat: {device.upper()}")
    
    # c. Inisialisasi Qdrant Client
    q_url = os.environ.get("QDRANT_URL")
    q_key = os.environ.get("QDRANT_API_KEY")
    if not q_url or not q_key:
        from google.colab import userdata
        q_url = userdata.get("QDRANT_URL").strip()
        q_key = userdata.get("QDRANT_API_KEY").strip()
    eval_qdrant = QdrantClient(url=q_url, api_key=q_key)
    
    # d. Ekstraksi dokumen PDF baru
    print(f"📄 Membaca dokumen baru: {TEST_PDF_PATH}...")
    doc = fitz.open(TEST_PDF_PATH)
    words_with_geometry = []
    
    total_chars = sum(len(page.get_text()) for page in doc)
    is_scanned = total_chars <= 50
    
    if is_scanned:
        print("⚠️ Catatan: Dokumen terdeteksi berupa Scanned PDF. Mengaktifkan EasyOCR parser...")
        if 'parser' in globals():
            parsed_data = parser.parse_pdf(TEST_PDF_PATH, "temp_sim")
            words_with_geometry = parsed_data["words"]
        else:
            print("❌ ERROR: Parser EasyOCR belum dimuat di memori Colab. Harap jalankan sel Langkah 4 terlebih dahulu.")
    else:
        for page_idx, page in enumerate(doc):
            words = page.get_text("words")
            for w in words:
                words_with_geometry.append({
                    "text": w[4],
                    "geometry": {"x1": w[0], "y1": w[1], "x2": w[2], "y2": w[3]},
                    "page": page_idx + 1
                })
                
    if not words_with_geometry:
        print("❌ Gagal: Tidak ada teks yang berhasil diekstraksi dari dokumen uji.")
    else:
        # e. Pemotongan teks (chunking)
        if 'text_embedder' in globals() and hasattr(text_embedder, 'chunk_text'):
            chunks = text_embedder.chunk_text(words_with_geometry)
        else:
            chunks = []
            chunk_size = 250
            for i in range(0, len(words_with_geometry), chunk_size - 50):
                chunk_words = words_with_geometry[i:i + chunk_size]
                if not chunk_words: break
                chunks.append({
                    "text": " ".join([w["text"] for w in chunk_words]),
                    "page": chunk_words[0]["page"]
                })
                
        print(f"✂️ Dokumen uji dipotong menjadi {len(chunks)} bagian semantik.")
        
        # f. Simulasi pengecekan kemiripan semantik per chunk
        print("\n=== MEMULAI SIMULASI PENGECEKAN KESAMAAN (Fine-Tuned SBERT) ===")
        high_similarity_found = False
        
        for idx, chunk in enumerate(chunks):
            query_text = f"query: {chunk['text']}"
            query_vector = eval_model.encode([query_text], show_progress_bar=False)[0].tolist()
            
            matches = eval_qdrant.search(
                collection_name="laporan_text_vectors",
                query_vector=qd_models.NamedVector(
                    name="text-dense",
                    vector=query_vector
                ),
                query_filter=qd_models.Filter(
                    must=[
                        qd_models.FieldCondition(
                            key="id_mata_kuliah",
                            match=qd_models.MatchValue(value=MATA_KULIAH_ID)
                        )
                    ]
                ),
                limit=1
            )
            
            if matches:
                best_match = matches[0]
                score = best_match.score
                
                # Tampilkan chunk tanpa batas threshold (score >= 0.0)
                if score >= 0.0:
                    print(f"\n📍 [Chunk {idx+1} - Halaman {chunk['page']}]")
                    print(f"   📝 Teks Uji: \"{chunk['text'][:100]}\"")
                    print(f"   🔥 Skor Kemiripan: {score * 100:.2f}%")
                    print(f"   📂 Cocok dengan Laporan ID: {best_match.payload.get('id_laporan')}")
                    print(f"   📖 Halaman Sumber: {best_match.payload.get('page')}")
                    print(f"   ✏️ Kutipan Sumber: \"{best_match.payload.get('text')[:100]}\"")
                    
                    if score >= 0.85:
                        high_similarity_found = True
                        
        print("\n===============================================================")
        if high_similarity_found:
            print("🚨 KESIMPULAN: Terdeteksi kemiripan ekstrim (>= 85%). Laporan baru berpotensi PLAGIAT.")
        else:
            print("✅ KESIMPULAN: Tidak ada indikasi kesamaan ekstrim. Laporan cenderung ORISINAL.")
        print("===============================================================")

---
## 🔍 Langkah 11: Analisis Silang Kemiripan Antar Laporan (Cross-Collusion Check)

Sel ini digunakan untuk menganalisis seluruh berkas PDF di dalam folder mata kuliah aktif ($WORKSPACE\_DIR$) untuk mendeteksi apakah ada laporan yang memiliki **kemiripan isi satu sama lain (saling menjiplak)** melampaui batas ambang kemiripan yang telah ditetapkan.

In [ ]:
import os
import re
from qdrant_client import QdrantClient
from qdrant_client.http import models as qd_models

# =========================================================================
# KONFIGURASI PENGECEKAN KEMIRIPAN SILANG (CROSS-COLLUSION)
# =========================================================================
# Batas kemiripan minimum untuk dilaporkan (contoh: 0.0 = 0% kemiripan untuk menampilkan semua)
SIMILARITY_THRESHOLD = 0.0
# =========================================================================

if 'MATA_KULIAH_ID' not in globals():
    MATA_KULIAH_ID = "77b21e8d-d779-46f3-a128-091a13be45cb"
    
if 'WORKSPACE_DIR' not in globals():
    WORKSPACE_DIR = "/content/drive/MyDrive/DATASET (cloud laporan matakuliah)/Laporan DPP 2025"

if not os.path.exists(WORKSPACE_DIR):
    print(f"❌ ERROR: Folder kerja tidak ditemukan: {WORKSPACE_DIR}")
else:
    pdf_files = [f for f in os.listdir(WORKSPACE_DIR) if f.endswith('.pdf')]
    print(f"📂 Ditemukan {len(pdf_files)} berkas PDF di folder untuk analisis silang.\n")
    
    # Sinkronisasi nama file dengan ID Laporan
    file_map = {}
    for filename in pdf_files:
        doc_id = re.sub(r'[^a-zA-Z0-9_]', '_', filename[:-4])
        file_map[doc_id] = filename

    # Inisialisasi Qdrant Client
    q_url = os.environ.get("QDRANT_URL")
    q_key = os.environ.get("QDRANT_API_KEY")
    if not q_url or not q_key:
        from google.colab import userdata
        q_url = userdata.get("QDRANT_URL").strip()
        q_key = userdata.get("QDRANT_API_KEY").strip()
    batch_qdrant = QdrantClient(url=q_url, api_key=q_key)

    # Membuat index payload secara aman sebelum pencarian filter untuk menghindari error Index required
    try:
        batch_qdrant.create_payload_index(
            collection_name="laporan_text_vectors",
            field_name="id_laporan",
            field_schema=qd_models.PayloadSchemaType.KEYWORD
        )
        batch_qdrant.create_payload_index(
            collection_name="laporan_text_vectors",
            field_name="id_mata_kuliah",
            field_schema=qd_models.PayloadSchemaType.KEYWORD
        )
    except Exception as e:
        pass

    print("⏳ Menjalankan analisis silang kemiripan antar laporan di Qdrant...")
    collusion_matches = []

    for idx, (doc_id, filename) in enumerate(file_map.items()):
        print(f"🔄 Menganalisis berkas [{idx+1}/{len(file_map)}]: {filename}...")
        
        try:
            # Ambil semua vektor teks yang terdaftar untuk laporan ini di Qdrant
            points, _ = batch_qdrant.scroll(
                collection_name="laporan_text_vectors",
                scroll_filter=qd_models.Filter(
                    must=[
                        qd_models.FieldCondition(
                            key="id_laporan",
                            match=qd_models.MatchValue(value=doc_id)
                        )
                    ]
                ),
                limit=1000,
                with_payload=True,
                with_vectors=True
            )
            
            if not points:
                print(f"   ⚠️ Berkas ini belum diindeks di Qdrant. Jalankan Langkah 8 terlebih dahulu.")
                continue
                
            for p in points:
                vectors_dict = p.vector
                if not vectors_dict or "text-dense" not in vectors_dict:
                    continue
                    
                chunk_vector = vectors_dict["text-dense"]
                source_page = p.payload.get("page", 1)
                source_text = p.payload.get("text", "")
                
                # Cari kecocokan di Qdrant dengan mengecualikan laporan itu sendiri
                matches = batch_qdrant.search(
                    collection_name="laporan_text_vectors",
                    query_vector=qd_models.NamedVector(
                        name="text-dense",
                        vector=chunk_vector
                    ),
                    query_filter=qd_models.Filter(
                        must=[
                            qd_models.FieldCondition(
                                key="id_mata_kuliah",
                                match=qd_models.MatchValue(value=MATA_KULIAH_ID)
                            )
                        ],
                        must_not=[
                            qd_models.FieldCondition(
                                key="id_laporan",
                                match=qd_models.MatchValue(value=doc_id)
                            )
                        ]
                    ),
                    limit=3
                )
                
                for match in matches:
                    score = match.score
                    if score >= SIMILARITY_THRESHOLD:
                        target_id = match.payload.get("id_laporan")
                        target_filename = file_map.get(target_id, target_id)
                        target_page = match.payload.get("page", 1)
                        target_text = match.payload.get("text", "")
                        
                        # Kunci pengenal pasangan dokumen agar urutannya konsisten
                        pair_key = tuple(sorted([doc_id, target_id]))
                        
                        collusion_matches.append({
                            "pair": pair_key,
                            "doc_a": filename,
                            "page_a": source_page,
                            "text_a": source_text,
                            "doc_b": target_filename,
                            "page_b": target_page,
                            "text_b": target_text,
                            "similarity": score
                        })
                        
        except Exception as e:
            print(f"   ❌ Gagal menganalisis {filename}. Error: {str(e)}")
            
    # Urutkan berdasarkan kemiripan tertinggi
    collusion_matches = sorted(collusion_matches, key=lambda x: x["similarity"], reverse=True)
    
    print("\n=========================================================================")
    print("📋 HASIL ANALISIS KEMISALAN SILANG ANTAR LAPORAN (CROSS-COLLUSION REPORT)")
    print(f"Kriteria Batas Kemiripan: >= {SIMILARITY_THRESHOLD * 100:.1f}%")
    print("=========================================================================")
    
    if not collusion_matches:
        print("✅ Bersih! Tidak ditemukan laporan yang memiliki indikasi kemiripan satu sama lain.")
    else:
        reported_pairs = set()
        match_count = 0
        
        for match in collusion_matches:
            pair = match["pair"]
            if pair in reported_pairs:
                continue
            reported_pairs.add(pair)
            match_count += 1
            
            print(f"\n🔥 [Kecocokan {match_count}] Tingkat Kemiripan: {match['similarity'] * 100:.2f}%")
            print(f"   📄 Laporan A: {match['doc_a']} (Halaman {match['page_a']})")
            print(f"      ↳ \"{match['text_a'][:120]}\"")
            print(f"   📄 Laporan B: {match['doc_b']} (Halaman {match['page_b']})")
            print(f"      ↳ \"{match['text_b'][:120]}\"")
            
        print(f"\nTotal ditemukan {match_count} pasangan laporan yang saling terindikasi memiliki kesamaan.")
    print("=========================================================================")